# FaceProof 02 — cache frozen features (CLIP + ResNet)

**Day 2–3.** Run every crop through two *frozen* backbones once and save the embeddings to Drive:
- **CLIP ViT-L/14** (768-d) → condition **C4** (the universal-detector hypothesis)
- **ImageNet ResNet-50** (2048-d) → condition **C1** (the baseline)

Frozen = we never train the backbone (that's the whole point of the Ojha reproduction, and it
keeps us inside free GPU limits). We cache so the probe in notebook 03 fits in seconds and we
never recompute. **Set Runtime ▸ Change runtime type ▸ GPU** first.

## 1. Setup

In [ ]:
# >>> SET THIS to your GitHub repo URL <<<
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "day1-preprocess-notebook"   # or "main" once the PR is merged

!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q open_clip_torch pyyaml tqdm

import sys, os, torch
sys.path.append(os.getcwd())
print("GPU available:", torch.cuda.is_available())   # want True — set Runtime > GPU

## 2. Mount Drive + paths

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'          # made in notebook 01
FEATURES_ROOT = ROOT / 'models' / 'features'     # cached features go here
MANIFEST      = ROOT / 'data' / 'manifest.csv'   # shared manifest (saved here so 03 reuses it)
FEATURES_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST.parent.mkdir(parents=True, exist_ok=True)
for d in ['real', 'stylegan', 'sd']:
    print(d, '->', len(list((CROPS_ROOT / d).glob('*.jpg'))), 'crops')

## 3. Rebuild the manifest (and save it to Drive)

Same deterministic build as notebook 01 (`seed=13`), so splits are identical. We save it to
Drive and extract features **in this exact row order** — that's how features and labels stay
aligned across notebooks.

In [ ]:
from src.data import build_manifest, make_splits

# Rebuild from the crops (deterministic) and SAVE to Drive so notebook 03 uses the EXACT same rows.
sources = {
    'real':     {'dir': str(CROPS_ROOT / 'real'),     'label': 0, 'generator': 'real'},
    'stylegan': {'dir': str(CROPS_ROOT / 'stylegan'), 'label': 1, 'generator': 'stylegan'},
    'sd':       {'dir': str(CROPS_ROOT / 'sd'),       'label': 1, 'generator': 'stable_diffusion'},
}
df = build_manifest(sources)
df = make_splits(df, train_generators=['stylegan'], test_unseen=['stable_diffusion'],
                 val_fraction=0.15, test_fraction=0.15, seed=13)
df.to_csv(MANIFEST, index=False)
print('saved manifest ->', MANIFEST, '| rows:', len(df))
paths = df['path'].tolist()   # feature arrays will be in THIS order

## 4. CLIP features (C4)

`extract_clip` (in `src/features.py`) loads CLIP, runs each crop through `encode_image`, and
L2-normalizes. Backbone/pretrained come from `configs/model.yaml` — no hard-coded settings.

In [ ]:
import yaml
from src.features import extract_clip

mcfg = yaml.safe_load(open('configs/model.yaml'))['clip']
# Extract once for ALL crops (aligned to `paths`), cache to Drive. Re-runs load instantly.
X_clip = extract_clip(paths,
                      backbone=mcfg['backbone'], pretrained=mcfg['pretrained'],
                      batch_size=64, cache=str(FEATURES_ROOT / 'clip_all.npy'))
print('CLIP features:', X_clip.shape)   # expect (N, 768)

## 5. ResNet features (C1)

In [ ]:
from src.features import extract_resnet

X_resnet = extract_resnet(paths, batch_size=64,
                          cache=str(FEATURES_ROOT / 'resnet_all.npy'))
print('ResNet features:', X_resnet.shape)   # expect (N, 2048)

## 6. ✅ Day 2–3 gate — features cached and aligned

In [ ]:
import numpy as np
assert X_clip.shape[0]   == len(df), 'CLIP rows must match manifest'
assert X_resnet.shape[0] == len(df), 'ResNet rows must match manifest'
assert X_clip.shape[1]   == 768
assert X_resnet.shape[1] == 2048
print('✅ Day 2 features cached and aligned to manifest:')
print(f'   CLIP   {X_clip.shape}  -> {FEATURES_ROOT / "clip_all.npy"}')
print(f'   ResNet {X_resnet.shape} -> {FEATURES_ROOT / "resnet_all.npy"}')
print('Next: notebook 03 — train the C4 (CLIP) probe and check in-distribution AUROC.')